In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_pay = pd.read_csv("data/03 Оплаты ХК.csv", sep=";")

In [ ]:
from general_information import read_actions, read_balances, read_payments
actions = read_actions()
payments = read_payments()
balance = read_balances()


In [ ]:
df_pay.head()

In [ ]:
df_pay.groupby(by="Номер",as_index=False).agg({"Сумма": "count"})

In [ ]:
from form_time_features import extract_payment_features

payment_features = extract_payment_features(payments, k=3, current_date=pd.to_datetime('2026-04-23'))
print(df_pay['Номер'].unique().shape, payment_features.shape)
print(f"Число неплатёжников: {(payment_features['Платежей_за_последние_3_мес'] == 0).sum()}")
payment_features.head()

In [ ]:
from form_time_features import calculate_complex_features

complex_features = calculate_complex_features(payments, balance, k=3, curr_date=pd.to_datetime('2026-01-23'))
complex_features.head()

In [ ]:
complex_features_cut = complex_features[complex_features['Days_Since_Clearance'] != 9999]
print(complex_features_cut.shape)
complex_features_cut.head()

In [ ]:
complex_features_cut["debt_current"] = 1

In [ ]:
from form_time_features import actions_features


act_feat = actions_features(complex_features_cut, actions, payments, balance, check_date='2025-11-23')
act_feat.iloc[:,8:].head()
# act_feat.columns

In [ ]:
import matplotlib.pyplot as plt

plt.hist(act_feat[act_feat.target > 0.01].target, bins = 70)

In [ ]:
act_feat.columns

In [ ]:
# Проверка число сезонных признаков
from form_time_features import get_seasonality_features

current_date=pd.to_datetime('2026-04-23')
df_season = get_seasonality_features(current_date)
df_season.head()

In [ ]:
import pandas as pd
import os

# путь к главному файлу
main_file = "data/14 Лимиты мер воздействия ХК.xlsx"

# читаем главный файл
limits_df = pd.read_excel(main_file)

# словарь для результатов
result = {}

for _, row in limits_df.iterrows():
    file_name = row.iloc[0]
    limit = row.iloc[1]

    # пропускаем пустые строки
    if pd.isna(file_name):
        continue

    file_path = os.path.join("data", file_name+".xlsx")

    # читаем файл без заголовков
    df_raw = pd.read_excel(file_path, header=None)

    # 1 строка — название операции
    operation_name = df_raw.iloc[0, 0]

    # 2 строка — заголовки
    df = df_raw.iloc[2:].copy()
    df.columns = df_raw.iloc[1]

    # чистка: убираем #Н/Д
    df = df[df["ЛС"].notna()]

    # сохраняем
    result[operation_name] = {
        "limit": limit,
        "data": df
    }

# теперь result — словарь с данными

In [ ]:
result.keys()

In [ ]:
result["Заявление о выдаче судебного приказа"]

In [ ]:
df2 = pd.read_excel("data/01 Общая информация о ЛС ХК.xlsx", index_col=0)
df2.drop(columns=["Адрес (ГУИД)"], inplace=True)
df2

In [ ]:
from general_information import read_general_information, read_balances, read_actions
from form_time_features import calculate_complex_features
import pandas as pd
# Читаем сальдовую ведомость и удаляем из неё нулевые строки
balances = read_balances()
cols_to_check = balances.columns.drop('ЛС')
balances = balances[(balances[cols_to_check] != 0).any(axis=1)]

# Все другие таблицы должны соответствовать данным id.
ids = balances['ЛС']

# Читаем платёжную таблицу. Удаляем лишние id.
df_pay = pd.read_csv("data/03 Оплаты ХК.csv", sep=";", decimal=",")
df_pay = df_pay[df_pay['Номер'].isin(ids)]

# Читаем информацию с булевыми признаками. Удаляем лишние id.
general_df = read_general_information()
general_df = general_df[general_df['ЛС'].isin(ids)]

actions_df = read_actions()

df = calculate_complex_features(df_pay, balances, 3, pd.Timestamp("2025-04-01"))
df

In [ ]:
df.Current_Debt.value_counts()

In [ ]:
from form_train_set import build_master_dataset

df = build_master_dataset(12)

df

In [ ]:
df.info()